# Lesson 3: Regression — Predicting Numbers

In Lesson 2 we did **classification**: the model looked at an image and picked a category. Now we tackle the other fundamental type of supervised learning — **regression**: predicting a continuous number.

How much will this house sell for? How long will the delivery take? What temperature will it be tomorrow? These are all regression problems. The output isn't a label, it's a number on a scale.

Let's see what that looks like with real data.

In [ ]:
# Colab setup - run this cell first if you're on Google Colab
try:
    import google.colab
    print("Colab environment ready! All required packages are pre-installed.")
except ImportError:
    pass  # Not on Colab, no action needed

In [ ]:
# === VISUALIZATION HELPERS (hidden to reduce clutter) ===
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def plot_gradient_descent_1d(f, df, x_start, lr, n_steps, x_range, title="Gradient Descent"):
    x = np.linspace(x_range[0], x_range[1], 200)
    y = f(x)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(x, y, 'b-', linewidth=2.5, label='Loss function')
    x_current = x_start
    positions = [x_current]
    for step in range(n_steps):
        y_current = f(x_current)
        gradient = df(x_current)
        color = plt.cm.Reds(0.3 + 0.7 * step / n_steps)
        ax.scatter([x_current], [y_current], color=color, s=100, zorder=5)
        tangent_x = np.linspace(x_current - 1.5, x_current + 1.5, 50)
        tangent_y = y_current + gradient * (tangent_x - x_current)
        ax.plot(tangent_x, tangent_y, '--', color=color, alpha=0.5, linewidth=1)
        x_new = x_current - lr * gradient
        positions.append(x_new)
        if abs(x_new - x_current) > 0.05:
            ax.annotate('', xy=(x_new, f(x_new)), xytext=(x_current, y_current),
                       arrowprops=dict(arrowstyle='->', color=color, lw=2))
        x_current = x_new
    ax.scatter([x_current], [f(x_current)], color='green', s=200, zorder=6,
               marker='*', label=f'Final: x={x_current:.2f}')
    ax.set_xlabel('Parameter value', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig, ax, positions

def plot_learning_rate_comparison(f, df, x_start, learning_rates, n_steps, x_range):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    x = np.linspace(x_range[0], x_range[1], 200)
    titles = ['Too Small', 'Just Right', 'Too Large']
    colors = ['blue', 'green', 'red']
    for ax, lr, title, color in zip(axes, learning_rates, titles, colors):
        ax.plot(x, f(x), 'b-', linewidth=2, alpha=0.5)
        x_current = x_start
        path_x, path_y = [x_current], [f(x_current)]
        for _ in range(n_steps):
            x_current = np.clip(x_current - lr * df(x_current), x_range[0], x_range[1])
            path_x.append(x_current)
            path_y.append(f(x_current))
        ax.plot(path_x, path_y, 'o-', color=color, markersize=8, linewidth=2)
        ax.scatter([path_x[0]], [path_y[0]], color='black', s=150, zorder=5, label='Start')
        ax.scatter([path_x[-1]], [path_y[-1]], color=color, s=150, zorder=5, marker='*', label='End')
        ax.set_title(f'{title}\nLR = {lr}', fontsize=12, fontweight='bold', color=color)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig, axes

def plot_loss_landscape(X_norm, y, predict_fn, loss_fn, w_range=(-1, 2), b_range=(0, 4)):
    weight_range = np.linspace(w_range[0], w_range[1], 50)
    bias_range = np.linspace(b_range[0], b_range[1], 50)
    W, B = np.meshgrid(weight_range, bias_range)
    loss_surface = np.zeros_like(W)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            loss_surface[i, j] = loss_fn(predict_fn(X_norm, W[i, j], B[i, j]), y)
    fig, ax = plt.subplots(figsize=(10, 8))
    contour = ax.contour(W, B, loss_surface, levels=30, cmap='viridis')
    ax.clabel(contour, inline=True, fontsize=8, fmt='%.1f')
    ax.set_xlabel('Weight (w)', fontsize=12)
    ax.set_ylabel('Bias (b)', fontsize=12)
    ax.set_title('Loss Landscape: Finding the Valley', fontsize=14)
    ax.scatter([0.80], [2.07], color='red', s=200, marker='*', zorder=5, label='Optimal point')
    ax.legend()
    plt.colorbar(contour, label='MSE Loss')
    plt.tight_layout()
    plt.show()

def plot_derivative_visual():
    def f(x): return x ** 2
    def df(x): return 2 * x
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    x = np.linspace(-3, 3, 100)
    points = [-2, 0, 1.5]
    titles = ['Steep downhill\n(negative slope)', 'Flat\n(zero slope)', 'Uphill\n(positive slope)']
    colors = ['red', 'green', 'blue']
    for ax, point, title, color in zip(axes, points, titles, colors):
        ax.plot(x, f(x), 'b-', linewidth=2, alpha=0.5)
        ax.scatter([point], [f(point)], color=color, s=150, zorder=5)
        slope = df(point)
        tangent_x = np.linspace(point - 1.5, point + 1.5, 50)
        tangent_y = f(point) + slope * (tangent_x - point)
        ax.plot(tangent_x, tangent_y, '--', color=color, linewidth=2.5, label=f'Slope = {slope:+.1f}')
        ax.set_xlim(-3.5, 3.5); ax.set_ylim(-1, 10)
        ax.set_title(title, fontsize=12, fontweight='bold', color=color)
        ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_training_progress(loss_history, weight_history, bias_history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, data, label, color, title in zip(
        axes, [loss_history, weight_history, bias_history],
        ['Loss (MSE)', 'Weight', 'Bias'], ['b', 'g', 'm'],
        ['Loss Decreases Over Training', 'Weight Converges', 'Bias Converges']):
        ax.plot(data, f'{color}-', linewidth=2)
        ax.set_xlabel('Iteration', fontsize=11); ax.set_ylabel(label, fontsize=11)
        ax.set_title(title, fontsize=12); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_model_fit(X_train, y_train, X_mean, X_std, weight, bias, predict_fn, sample_n=1000):
    fig, ax = plt.subplots(figsize=(10, 6))
    idx = np.random.choice(len(X_train), min(sample_n, len(X_train)), replace=False)
    ax.scatter(X_train[idx], y_train[idx], alpha=0.3, s=20, label='Training data')
    x_line = np.linspace(X_train.min(), X_train.max(), 100)
    y_line = predict_fn((x_line - X_mean) / X_std, weight, bias)
    ax.plot(x_line, y_line, 'r-', linewidth=2.5, label='Learned linear model')
    ax.set_xlabel('Median Income ($10,000s)', fontsize=12)
    ax.set_ylabel('House Value ($100,000s)', fontsize=12)
    ax.set_title('Gradient Descent Found the Best Fit Line!', fontsize=14)
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_early_stopping_comparison(train_losses, test_losses):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].plot(train_losses, 'b-', linewidth=2, label='Training Loss')
    axes[0].plot(test_losses, 'orange', linewidth=2, linestyle='--', label='Test Loss')
    axes[0].set_xlabel('Iteration', fontsize=12); axes[0].set_ylabel('Loss (MSE)', fontsize=12)
    axes[0].set_title('Our Model: No Overfitting', fontsize=12, color='green')
    axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)
    iters = np.arange(1, 101)
    s_train = 2.5 * np.exp(-0.06 * iters) + 0.3
    s_test = s_train + 0.015 * (iters - 30) * (iters > 30)
    axes[1].plot(iters, s_train, 'b-', linewidth=2, label='Training Loss')
    axes[1].plot(iters, s_test, 'orange', linewidth=2, linestyle='--', label='Test Loss')
    axes[1].axvline(x=30, color='red', linestyle=':', linewidth=2, alpha=0.7)
    axes[1].annotate('Stop here!', xy=(30, s_test[29]), xytext=(50, 2.0),
                    fontsize=12, fontweight='bold', color='red',
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
    axes[1].fill_between(iters[29:], s_train[29:], s_test[29:], alpha=0.2, color='red', label='Overfitting gap')
    axes[1].set_xlabel('Iteration', fontsize=12); axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].set_title('What Overfitting Looks Like', fontsize=12, color='red')
    axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_polynomial_fits(x_data, y_data, x_plot_range=(0, 8), degrees=[1, 3, 12],
                         titles=None, colors=None, ylim=(-2, 7)):
    if titles is None: titles = [f'Degree {d}' for d in degrees]
    if colors is None: colors = ['red', 'green', 'purple']
    x_plot = np.linspace(x_plot_range[0], x_plot_range[1], 300)
    fig, axes = plt.subplots(1, len(degrees), figsize=(16, 5))
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import LinearRegression
    for ax, degree, title, color in zip(axes, degrees, titles, colors):
        ax.scatter(x_data, y_data, color='black', s=60, zorder=5, label='Training data')
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        X_poly = poly.fit_transform(x_data.reshape(-1, 1))
        model = LinearRegression().fit(X_poly, y_data)
        y_plot = np.clip(model.predict(poly.transform(x_plot.reshape(-1, 1))), ylim[0]-1, ylim[1]+1)
        ax.plot(x_plot, y_plot, color=color, linewidth=2.5)
        ax.set_ylim(ylim); ax.set_xlabel('x'); ax.set_ylabel('y')
        ax.set_title(title, fontsize=13, fontweight='bold', color=color)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_complexity_curve(degrees, train_mses, test_mses):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(degrees, train_mses, 'b-o', linewidth=2, markersize=8, label='Train MSE')
    ax.plot(degrees, test_mses, 'r--s', linewidth=2, markersize=8, label='Test MSE')
    best_idx = np.argmin(test_mses)
    best_degree = degrees[best_idx]
    ax.scatter([best_degree], [test_mses[best_idx]], color='green', s=200, zorder=5,
               marker='*', label=f'Best: degree {best_degree}')
    if best_degree < max(degrees):
        ax.axvspan(best_degree + 0.5, max(degrees) + 0.5, alpha=0.1, color='red', label='Overfitting zone')
    ax.set_xlabel('Polynomial Degree (model complexity)', fontsize=12)
    ax.set_ylabel('MSE', fontsize=12)
    ax.set_title('The Overfitting Curve: More Complexity ≠ Better', fontsize=14)
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3); ax.set_xticks(degrees)
    plt.tight_layout()
    plt.show()
    return best_degree

def plot_extrapolation(x_data, y_data, x_range=(-1, 10), degrees=[3, 12],
                       colors=['green', 'purple'], train_range=(0.5, 7.5)):
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import LinearRegression
    x_ext = np.linspace(x_range[0], x_range[1], 500)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(x_data, y_data, color='black', s=80, zorder=5, label='Training data')
    for degree, color in zip(degrees, colors):
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        model = LinearRegression().fit(poly.fit_transform(x_data.reshape(-1, 1)), y_data)
        y_ext = model.predict(poly.transform(x_ext.reshape(-1, 1)))
        ax.plot(x_ext, y_ext, color=color, linewidth=2.5, label=f'Degree {degree}')
    ax.axvspan(x_range[0], train_range[0], alpha=0.15, color='red', label='Extrapolation zone')
    ax.axvspan(train_range[1], x_range[1], alpha=0.15, color='red')
    ax.axvline(x=train_range[0], color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x=train_range[1], color='gray', linestyle=':', alpha=0.5)
    ax.set_ylim(-5, 10); ax.set_xlabel('x', fontsize=12); ax.set_ylabel('y', fontsize=12)
    ax.set_title('Extrapolation: Overfit Models Explode Outside Training Range', fontsize=14)
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_feature_bars(names, weights, title='Feature Importance', noise_prefix=None):
    sorted_idx = np.argsort(np.abs(weights))
    sorted_names = np.array(names)[sorted_idx]
    sorted_weights = weights[sorted_idx]
    colors = []
    for name, w in zip(sorted_names, sorted_weights):
        if noise_prefix and noise_prefix in str(name):
            colors.append('#95a5a6')
        elif w < 0:
            colors.append('#e74c3c')
        else:
            colors.append('#2ecc71')
    fig, ax = plt.subplots(figsize=(10, max(4, len(names) * 0.45)))
    ax.barh(sorted_names, sorted_weights, color=colors)
    ax.set_xlabel('Weight (on standardized features)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

def plot_r2_progression(labels, scores):
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6'][:len(labels)]
    bars = ax.barh(labels, scores, color=colors)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=12, fontweight='bold')
    ax.set_xlabel('R² (proportion of variance explained)', fontsize=12)
    ax.set_title('More Relevant Features → Better Predictions', fontsize=14)
    ax.set_xlim(0, 0.75); ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()


def plot_regression_intro(income, price, sample_n=1000):
    rng = np.random.RandomState(42)
    idx = rng.choice(len(income), min(sample_n, len(income)), replace=False)
    x_s, y_s = income[idx], price[idx]
    x_line = np.linspace(0, 15, 100)
    lines = [
        (0.0, 2.1, 'Flat line — ignores income', '#e74c3c'),
        (0.15, 1.5, 'Some slope — getting warmer', '#e67e22'),
        (0.42, 0.45, 'Best fit — captures the trend', '#2ecc71'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, (slope, intercept, title, color) in zip(axes, lines):
        ax.scatter(x_s, y_s, alpha=0.2, s=10, color='steelblue')
        ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=3)
        ax.set_xlabel('Median Income ($10,000s)', fontsize=11)
        ax.set_ylabel('House Value ($100,000s)', fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold', color=color)
        ax.set_ylim(0, 5.5)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def draw_batch_gd_diagram(X_data, y_data, weights, bias, feature_names, n_total_rows):
    predictions = X_data @ weights + bias
    errors = predictions - y_data
    n_rows = len(X_data)
    n_feat = len(feature_names)

    fig, ax = plt.subplots(figsize=(15, 7))
    ax.axis('off')
    ax.set_xlim(-1.5, 15.5)
    ax.set_ylim(-2, 8)

    cw, ch = 1.8, 0.7
    CELL_BG = '#f7f7f7'
    CELL_EC = '#555555'
    BRACKET_COLOR = '#333333'

    def draw_bracket(ax, x, y_top, y_bot, side='left'):
        bw = 0.15
        sign = -1 if side == 'left' else 1
        ax.plot([x + sign*bw, x, x, x + sign*bw],
                [y_top, y_top, y_bot, y_bot],
                color=BRACKET_COLOR, lw=1.8, solid_capstyle='round')

    # ── Weight vector (horizontal, top) ──
    wx, wy = 0.5, 6.5
    ax.text(wx + n_feat*cw/2, wy + ch + 0.25, 'weights', ha='center', fontsize=11, fontweight='bold')
    draw_bracket(ax, wx - 0.05, wy + ch, wy, side='left')
    draw_bracket(ax, wx + n_feat*cw + 0.05, wy + ch, wy, side='right')
    for j in range(n_feat):
        rect = plt.Rectangle((wx + j*cw, wy), cw - 0.08, ch,
                             facecolor=CELL_BG, edgecolor=CELL_EC, linewidth=1)
        ax.add_patch(rect)
        ax.text(wx + j*cw + cw/2, wy + ch/2, f'{weights[j]:+.3f}',
                ha='center', va='center', fontsize=10, family='monospace')
    # Feature labels below weight cells
    for j, name in enumerate(feature_names):
        ax.text(wx + j*cw + cw/2, wy - 0.2, name,
                ha='center', va='top', fontsize=7, color='#888888')

    # Bias cell
    bx = wx + n_feat*cw + 0.8
    rect = plt.Rectangle((bx, wy), cw*0.65, ch,
                         facecolor=CELL_BG, edgecolor=CELL_EC, linewidth=1)
    ax.add_patch(rect)
    ax.text(bx + cw*0.325, wy + ch/2, f'{bias:.3f}',
            ha='center', va='center', fontsize=10, family='monospace')
    ax.text(bx + cw*0.325, wy + ch + 0.25, 'bias', ha='center', fontsize=10, fontweight='bold')

    # ── Data matrix (left) ──
    dx, dy_start = 0.5, 3.5
    row_gap = 1.5
    ax.text(dx + n_feat*cw/2, dy_start + ch + 0.25, 'X  (data matrix)', ha='center', fontsize=11, fontweight='bold')
    # Column headers
    for j, name in enumerate(feature_names):
        ax.text(dx + j*cw + cw/2, dy_start + ch + 0.05, name,
                ha='center', va='bottom', fontsize=7, color='#888888')

    for i in range(n_rows):
        row_y = dy_start - i * row_gap
        # Left bracket
        draw_bracket(ax, dx - 0.05, row_y + ch, row_y, side='left')
        draw_bracket(ax, dx + n_feat*cw + 0.05, row_y + ch, row_y, side='right')
        # Row label
        ax.text(dx - 0.35, row_y + ch/2, f'row {i+1}',
                ha='right', va='center', fontsize=8, color='#999999')
        for j in range(n_feat):
            rect = plt.Rectangle((dx + j*cw, row_y), cw - 0.08, ch,
                                 facecolor=CELL_BG, edgecolor=CELL_EC, linewidth=0.8)
            ax.add_patch(rect)
            ax.text(dx + j*cw + cw/2, row_y + ch/2, f'{X_data[i,j]:+.2f}',
                    ha='center', va='center', fontsize=9.5, family='monospace')

        # ── Arrows from each weight cell down to the corresponding data cell ──
        for j in range(n_feat):
            w_center_x = wx + j*cw + cw/2
            d_center_x = dx + j*cw + cw/2
            ax.annotate('',
                xy=(d_center_x, row_y + ch + 0.03),
                xytext=(w_center_x, wy - 0.03),
                arrowprops=dict(arrowstyle='->', color='#bbbbbb', lw=0.7,
                               connectionstyle=f'arc3,rad={0.05 * (j - 1)}'))

        # ── Horizontal arrow to prediction ──
        arrow_x = dx + n_feat*cw + 0.2
        arrow_end = 8.8
        ax.annotate('', xy=(arrow_end, row_y + ch/2), xytext=(arrow_x, row_y + ch/2),
                    arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3))
        mid_x = (arrow_x + arrow_end) / 2
        ax.text(mid_x, row_y + ch/2 + 0.22, 'row \u00b7 w + b',
                ha='center', va='bottom', fontsize=7.5, color='#888888', style='italic')

        # ── Prediction ──
        px = arrow_end + 0.15
        rect = plt.Rectangle((px, row_y), cw*0.75, ch,
                             facecolor=CELL_BG, edgecolor=CELL_EC, linewidth=1)
        ax.add_patch(rect)
        ax.text(px + cw*0.375, row_y + ch/2, f'{predictions[i]:+.3f}',
                ha='center', va='center', fontsize=9.5, family='monospace')

        # ── "- actual = error" ──
        tx = px + cw*0.75 + 0.15
        ax.text(tx, row_y + ch/2, f'\u2212 {y_data[i]:.2f}  =',
                ha='left', va='center', fontsize=9, family='monospace', color='#888888')
        ex = tx + 2.3
        ax.text(ex, row_y + ch/2, f'{errors[i]:+.3f}',
                ha='left', va='center', fontsize=9.5, family='monospace',
                fontweight='bold', color='#c0392b')

    # "..." for remaining rows
    dots_y = dy_start - n_rows * row_gap + 0.3
    ax.text(dx + n_feat*cw/2, dots_y, f'\u22ee  {n_total_rows - n_rows:,} more rows  \u22ee',
            ha='center', fontsize=9, color='#aaaaaa')

    # Column labels
    ax.text(9.6, dy_start + ch + 0.25, '\u0177', fontsize=11, fontweight='bold', ha='center')
    ax.text(13.5, dy_start + ch + 0.25, 'error', fontsize=10, fontweight='bold', color='#c0392b', ha='center')

    # ── Loss ──
    loss_val = (errors**2).mean()
    loss_x, loss_y = 11.5, dots_y - 0.8
    rect = plt.Rectangle((loss_x, loss_y), 3.2, 0.75,
                         facecolor='white', edgecolor='#c0392b', linewidth=2)
    ax.add_patch(rect)
    ax.text(loss_x + 1.6, loss_y + 0.375, f'MSE = {loss_val:.4f}',
            ha='center', va='center', fontsize=11, fontweight='bold', color='#c0392b')
    ax.annotate('', xy=(loss_x + 1.6, loss_y + 0.75),
                xytext=(loss_x + 1.6, dots_y + 0.5),
                arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.3))
    ax.text(loss_x + 1.6, dots_y + 0.55, 'mean(errors\u00b2)',
            ha='center', va='bottom', fontsize=8, color='#c0392b', style='italic')

    plt.tight_layout()
    plt.show()
    return predictions, errors

def plot_error_comparison(x_data, y_data, fits):
    """2-panel comparison showing error lines for different fits.
    fits: list of (slope, intercept, title, color) tuples."""
    fig, axes = plt.subplots(1, len(fits), figsize=(7*len(fits), 5))
    if len(fits) == 1:
        axes = [axes]
    mse_values = []
    for ax, (slope, intercept, title, color) in zip(axes, fits):
        ax.scatter(x_data, y_data, color='steelblue', s=60, zorder=5)
        x_line = np.linspace(x_data.min() - 0.5, x_data.max() + 0.5, 100)
        ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=2.5)
        total_sq_error = 0
        for xi, yi in zip(x_data, y_data):
            pred_i = slope * xi + intercept
            ax.plot([xi, xi], [yi, pred_i], color='red', linewidth=1.5, alpha=0.6)
            total_sq_error += (pred_i - yi) ** 2
        mse = total_sq_error / len(x_data)
        mse_values.append(mse)
        ax.set_xlabel('Median Income ($10,000s)', fontsize=11)
        ax.set_ylabel('House Value ($100,000s)', fontsize=11)
        ax.set_title(f'{title}\nMSE = {mse:.2f}', fontsize=12, fontweight='bold', color=color)
        ax.set_ylim(0, 5.5); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return mse_values

def plot_scaling_landscape(X_raw, X_norm, y, predict_fn, loss_fn):
    """2-panel contour: unscaled (elongated) vs scaled (round) loss landscape with GD paths."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Unscaled
    w_vals = np.linspace(-0.1, 0.7, 50)
    b_vals = np.linspace(-0.5, 2.5, 50)
    W_grid, B_grid = np.meshgrid(w_vals, b_vals)
    loss_raw = np.array([[loss_fn(predict_fn(X_raw, W_grid[i,j], B_grid[i,j]), y)
                          for j in range(50)] for i in range(50)])
    w, b = 0.0, 0.0
    path_w, path_b = [w], [b]
    for _ in range(150):
        errors = predict_fn(X_raw, w, b) - y
        n = len(y)
        w -= 0.001 * (2/n) * (errors * X_raw).sum()
        b -= 0.001 * (2/n) * errors.sum()
        path_w.append(w); path_b.append(b)
    axes[0].contour(W_grid, B_grid, loss_raw, levels=25, cmap='viridis', alpha=0.8)
    axes[0].plot(path_w, path_b, 'r.-', markersize=2, linewidth=1.5, alpha=0.8)
    axes[0].scatter(path_w[0], path_b[0], color='red', s=100, zorder=5, label='Start')
    axes[0].scatter(path_w[-1], path_b[-1], color='#e74c3c', s=150, marker='*', zorder=5, label='After 150 steps')
    axes[0].set_xlabel('Weight', fontsize=11); axes[0].set_ylabel('Bias', fontsize=11)
    axes[0].set_title('Unscaled Features\nElongated valley, indirect path',
                       fontsize=11, fontweight='bold', color='#e74c3c')
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.2)

    # Right: Scaled
    w_vals_n = np.linspace(-1, 2, 50)
    b_vals_n = np.linspace(-0.5, 4, 50)
    W_n, B_n = np.meshgrid(w_vals_n, b_vals_n)
    loss_norm = np.array([[loss_fn(predict_fn(X_norm, W_n[i,j], B_n[i,j]), y)
                           for j in range(50)] for i in range(50)])
    w, b = 0.0, 0.0
    path_w, path_b = [w], [b]
    for _ in range(20):
        errors = predict_fn(X_norm, w, b) - y
        n = len(y)
        w -= 0.1 * (2/n) * (errors * X_norm).sum()
        b -= 0.1 * (2/n) * errors.sum()
        path_w.append(w); path_b.append(b)
    axes[1].contour(W_n, B_n, loss_norm, levels=25, cmap='viridis', alpha=0.8)
    axes[1].plot(path_w, path_b, 'r.-', markersize=4, linewidth=1.5, alpha=0.8)
    axes[1].scatter(path_w[0], path_b[0], color='red', s=100, zorder=5, label='Start')
    axes[1].scatter(path_w[-1], path_b[-1], color='#2ecc71', s=150, marker='*', zorder=5, label='After 20 steps')
    axes[1].set_xlabel('Weight', fontsize=11); axes[1].set_ylabel('Bias', fontsize=11)
    axes[1].set_title('Scaled Features\nRound valley, direct path',
                       fontsize=11, fontweight='bold', color='#2ecc71')
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

def plot_scaling_convergence(configs, y, predict_fn, loss_fn, n_iters=100):
    """Convergence curves for different scaling/lr combos.
    configs: list of (label, X_data, lr, color, linestyle) tuples."""
    fig, ax = plt.subplots(figsize=(12, 6))
    for label, X_data, lr, color, style in configs:
        w, b, losses = 0.0, 0.0, []
        for _ in range(n_iters):
            preds = predict_fn(X_data, w, b)
            loss = loss_fn(preds, y)
            losses.append(loss)
            if loss > 1e10:
                break
            errors = preds - y
            n = len(y)
            w -= lr * (2/n) * (errors * X_data).sum()
            b -= lr * (2/n) * errors.sum()
        ax.plot(range(len(losses)), losses, color=color, linewidth=2.5,
                linestyle=style, label=label)
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Loss (MSE)', fontsize=12)
    ax.set_title('Why Feature Scaling Matters', fontsize=14)
    ax.legend(fontsize=11)
    ax.set_ylim(0, 12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_actual_vs_predicted(y_true, predictions_list, labels, r2_scores, colors):
    """Side-by-side actual vs predicted scatter plots."""
    n = len(predictions_list)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 5))
    if n == 1:
        axes = [axes]
    lims = [min(y_true.min(), min(p.min() for p in predictions_list)) - 2,
            max(y_true.max(), max(p.max() for p in predictions_list)) + 2]
    for ax, preds, label, r2, color in zip(axes, predictions_list, labels, r2_scores, colors):
        ax.scatter(y_true, preds, alpha=0.6, s=35, color=color)
        ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect predictions')
        ax.set_xlabel('Actual', fontsize=11)
        ax.set_ylabel('Predicted', fontsize=11)
        ax.set_title(f'{label}: R\u00b2 = {r2:.2f}', fontsize=12, fontweight='bold')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("Visualization helpers loaded!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

DATA_PATH = Path("../../../../data")  # Change if your data is elsewhere

print("Ready to learn about gradient descent!")

## The Core Idea: Fitting a Line to Data

Here's 20,000 California housing districts. Each dot is a district — its x position is the median income, its y position is the median house value. There's clearly a relationship: richer districts have pricier homes.

Regression = finding the line (or curve) that best captures that relationship. Three attempts:

In [ ]:
# Load California Housing data and show three regression attempts
from sklearn.datasets import fetch_california_housing
_housing = fetch_california_housing()
plot_regression_intro(_housing.data[:, 0], _housing.target)

print("Left:   Ignores income entirely — terrible predictions")
print("Middle: Tilted in the right direction — but still off")
print("Right:  Captures the upward trend — the 'best fit' line")

The model is just: **prediction = weight × income + bias**. That's two numbers — the slope and the intercept. The right panel found good values; the left two didn't.

The entire rest of this notebook is about answering one question: **how do we find those best values automatically?** That's where loss functions, gradients, and gradient descent come in. But first, let's set up the data properly.

## Part 1: The California Housing Dataset

We already saw the income-vs-price scatter above. Let's look at the full dataset — 20,640 districts, 8 features each. We'll start with just income as our single input, then use all 8 features later in Part 11.

In [ ]:
# Load the California Housing dataset
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)
df['MedHouseVal'] = california.target

print("California Housing Dataset")
print("=" * 60)
print(f"Number of districts: {len(df):,}")
print(f"Features: {california.feature_names}")
print(f"\nTarget: Median House Value (in $100,000s)")
print(f"  Range: ${df['MedHouseVal'].min()*100:.0f}k - ${df['MedHouseVal'].max()*100:.0f}k")
print(f"\nWith {len(df):,} samples, we have plenty of data for learning!")

df.head()

### Train/Test Split: The Golden Rule

**Never evaluate your model on the same data you trained it on.**

Why? Because a model can memorize training data perfectly but fail completely on new examples. We need to test on data the model has never seen.

We split our data into:
- **Training set** (~80%): Used to learn the parameters
- **Test set** (~20%): Held back completely, used only for final evaluation

This is the most fundamental rule in machine learning validation.

In [ ]:
# Focus on ONE feature first: median income
# This makes visualization easy and concepts clear
X = df['MedInc'].values
y = df['MedHouseVal'].values

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Total districts:  {len(X):,}")
print(f"Training set:     {len(X_train):,} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set:         {len(X_test):,} ({len(X_test)/len(X)*100:.0f}%)")
print("\nWe'll train on 16,512 districts and test on 4,128 unseen districts.")

## Part 2: Measuring "How Wrong" — The Loss Function

We found a good line by eye in the intro. But a computer can't eyeball a scatter plot. It needs a **number** that says how bad a given line is — then it can try to make that number smaller.

The natural idea: for each data point, measure the vertical distance between where the line predicts and where the point actually is. Big gaps = bad fit. Small gaps = good fit.

In [ ]:
# Sample 20 districts so the error lines are visible
np.random.seed(42)
_idx = np.random.choice(len(X_train), 20, replace=False)
x_sample, y_sample = X_train[_idx], y_train[_idx]

fits = [
    (0.0, 2.1, 'Bad fit: ignores the trend', '#e74c3c'),
    (0.42, 0.45, 'Good fit: follows the trend', '#2ecc71'),
]
mse_bad, mse_good = plot_error_comparison(x_sample, y_sample, fits)

print("Red lines = errors. Each one is (prediction - actual) for one data point.")
print(f"Bad fit MSE:  {mse_bad:.2f} — big gaps everywhere")
print(f"Good fit MSE: {mse_good:.2f} — much smaller gaps")

Those red lines are the **errors** (also called **residuals**). We need to collapse all of them into a single number. That's the **loss function**.

For regression, the standard choice is **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\text{prediction}_i - \text{actual}_i)^2$$

Why squared? Squaring does two things: it makes all errors positive (undershooting and overshooting both count), and it punishes big misses disproportionately — predicting $300k for a $500k house is more than twice as bad as being off by $100k.

**Training = finding parameters that minimize this number.**

MSE isn't the only option. Different problems call for different loss functions: classification (Lesson 2) uses **cross-entropy** because the output is a probability, not a dollar amount. Even within regression, **MAE** (mean absolute error) treats all errors equally instead of punishing outliers. The choice of loss function shapes what the model optimizes for — but MSE is the default starting point for regression, and it's what we'll use here.

In [ ]:
def predict(x, weight, bias):
    """Linear prediction: y = wx + b"""
    return weight * x + bias

def mse_loss(predictions, actual):
    """Mean Squared Error — our loss function"""
    return ((predictions - actual) ** 2).mean()

# Normalize the input features for stable gradient descent
# This is CRITICAL — we'll see why in Part 6
X_mean, X_std = X_train.mean(), X_train.std()
X_train_norm = (X_train - X_mean) / X_std
X_test_norm = (X_test - X_mean) / X_std

# Quick check: what's the MSE for our good-fit line from above?
good_preds = predict(X_train_norm, 0.80, 2.07)
print(f"MSE with (w=0.80, b=2.07): {mse_loss(good_preds, y_train):.4f}")
print(f"MSE with (w=0.00, b=0.00): {mse_loss(predict(X_train_norm, 0, 0), y_train):.4f}")
print("\nLower is better. Can we find the parameters that give the absolute lowest MSE?")

In [ ]:
# Visualize the loss for every (w, b) combination
plot_loss_landscape(X_train_norm, y_train, predict, mse_loss)

print("The loss surface forms a bowl shape.")
print("Our goal: find the parameters at the bottom of this bowl!")

### Can You Find the Best Parameters by Hand?

The contour plot above shows the MSE for every (weight, bias) combination. The dark center is the valley — the best parameters live there.

Before we automate anything, try it yourself. Use the sliders below to adjust the weight and bias. The left panel shows your line on the data, the right panel shows where you are on the loss landscape. Try to get the lowest MSE you can.

In [ ]:
from ipywidgets import FloatSlider, VBox, HBox, interactive_output
from IPython.display import display, Image
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import io

# Sample data for faster interactive plotting
sample_idx_interactive = np.random.choice(len(X_train), 800, replace=False)

# Pre-compute loss landscape grid (once, not on every slider move)
_wr = np.linspace(-1, 2, 40)
_br = np.linspace(-1, 5, 40)
_W, _B = np.meshgrid(_wr, _br)
_loss_grid = np.array([[mse_loss(predict(X_train_norm, _W[i, j], _B[i, j]), y_train)
                         for j in range(_W.shape[1])] for i in range(_W.shape[0])])

weight_slider = FloatSlider(min=-1.0, max=2.0, step=0.05, value=0.0,
    description='Weight (w):', style={'description_width': '100px'}, layout={'width': '500px'})
bias_slider = FloatSlider(min=-1.0, max=5.0, step=0.05, value=0.0,
    description='Bias (b):', style={'description_width': '100px'}, layout={'width': '500px'})

def plot_interactive_fit(w, b):
    loss = mse_loss(predict(X_train_norm, w, b), y_train)

    # OO matplotlib API — completely bypasses pyplot's figure registry and display hooks
    fig = Figure(figsize=(14, 5))
    FigureCanvasAgg(fig)
    axes = fig.subplots(1, 2)

    axes[0].scatter(X_train[sample_idx_interactive], y_train[sample_idx_interactive], alpha=0.3, s=15)
    x_line = np.linspace(X_train.min(), X_train.max(), 100)
    axes[0].plot(x_line, predict((x_line - X_mean) / X_std, w, b), 'r-', linewidth=2.5)
    axes[0].set_xlabel('Median Income ($10,000s)', fontsize=11)
    axes[0].set_ylabel('House Value ($100,000s)', fontsize=11)
    axes[0].set_title(f'w = {w:.2f}, b = {b:.2f}', fontsize=12)
    axes[0].set_ylim(-0.5, 5.5); axes[0].grid(True, alpha=0.3)

    axes[1].contourf(_W, _B, _loss_grid, levels=25, cmap='viridis', alpha=0.8)
    axes[1].scatter([w], [b], color='red', s=200, marker='X', zorder=5,
                    edgecolors='white', linewidth=2)
    axes[1].set_xlabel('Weight (w)', fontsize=11)
    axes[1].set_ylabel('Bias (b)', fontsize=11)
    axes[1].set_title(f'MSE = {loss:.4f}', fontsize=12,
                      color='green' if loss < 0.75 else 'red')
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    display(Image(data=buf.getvalue()))

out = interactive_output(plot_interactive_fit, {'w': weight_slider, 'b': bias_slider})
display(VBox([HBox([weight_slider, bias_slider]), out]))

### From Manual to Automatic

You just optimized a model by hand — tweaking two sliders until the MSE stopped going down. You probably got close to the optimal point (around w=0.80, b=2.07), but it took some fiddling.

Now imagine doing this with 8 features instead of 1. That's 9 sliders (one weight per feature plus a bias). Or a neural network with 10 million parameters. You can't hand-tune 10 million sliders.

We need an algorithm that can **automatically** walk downhill on the loss landscape until it reaches the bottom. That algorithm is **gradient descent**. But first, we need one mathematical tool: how do you figure out which direction is "downhill" at any given point?

## Part 3: Which Direction is Downhill? — The Derivative

The answer is the **derivative** (or **gradient** when you have multiple parameters). It tells you the slope of the loss at your current position — how steep the landscape is and which way it tilts.

Think about standing on a hill blindfolded. You can feel the ground sloping under your feet. That slope IS the derivative:

- **Negative slope**: The ground drops to the right — move right to go downhill
- **Positive slope**: The ground rises to the right — move left to go downhill
- **Zero slope**: You're at a flat spot — you might be at the bottom!

In [ ]:
# Visual explanation: the derivative is the slope at each point
def simple_curve(x):
    return x ** 2

def derivative_of_curve(x):
    return 2 * x  # The derivative of x² is 2x

plot_derivative_visual()

print("Key insight: At the MINIMUM, the slope (derivative) is zero!")
print("The derivative tells us which direction is 'downhill'.")

## Part 4: Gradient Descent — The Algorithm

Now we have everything we need. The **gradient descent** algorithm is:

$$w_{\text{new}} = w_{\text{old}} - \text{learning\_rate} \times \text{gradient}$$

Let's break this down:
- **gradient**: The slope/derivative — tells us direction of steepest increase
- **learning_rate**: How big of a step to take
- **minus sign**: Move OPPOSITE to the gradient (go downhill, not uphill!)

It's like a ball rolling downhill — it follows the steepest descent until it reaches the bottom.

In [ ]:
# Demonstrate gradient descent on a simple parabola
fig, ax, positions = plot_gradient_descent_1d(
    f=simple_curve,
    df=derivative_of_curve,
    x_start=-2.5,
    lr=0.1,
    n_steps=10,
    x_range=(-3, 3),
    title="Gradient Descent: Rolling Downhill to the Minimum"
)
plt.show()

print(f"Started at x = {positions[0]:.2f}")
print(f"After 10 steps: x = {positions[-1]:.2f}")
print("\nEach colored dot is one step. Dashed lines show the tangent (gradient).")
print("The ball follows the slope downhill until it reaches the minimum!")

### Implement It Yourself

The visualization above shows 10 steps of gradient descent. Let's write those same steps as code — the actual algorithm is just 3 lines inside a loop:

In [ ]:
# Gradient descent in 3 lines
x_position = -2.5
learning_rate = 0.1

print(f"{'Step':>4}  {'x':>8}  {'Loss':>8}  {'Gradient':>10}")
print(f"   0  {x_position:>8.4f}  {simple_curve(x_position):>8.4f}")

for step in range(1, 11):
    gradient = derivative_of_curve(x_position)              # 1. Compute gradient
    x_position = x_position - learning_rate * gradient      # 2. Step opposite to gradient
    print(f"{step:>4}  {x_position:>8.4f}  {simple_curve(x_position):>8.4f}  {gradient:>+10.4f}")

print(f"\nFrom x = -2.50 (loss = 6.25) to x = {x_position:.4f} (loss = {simple_curve(x_position):.4f})")
print(f"Three lines: compute gradient, subtract lr × gradient, repeat.")

### Scaling Up — Batch Gradient Descent in Plain English

We just did gradient descent on a single number. Real training works the same way, just with more numbers at once. Here's the whole process:

1. **Make up some weights.** One random number per feature, plus a bias. These start as garbage.
2. **Multiply every data row by those weights.** Each row gets dot-producted with the weight vector, bias added. One prediction per row. With random weights, these predictions are terrible.
3. **Measure how wrong we are.** Compare predictions to actual values, square the differences, average them. That's the MSE loss.
4. **Figure out which direction to nudge each weight.** That's the gradient. For each weight: "if I increase this, does the loss go up or down?"
5. **Nudge the weights.** Subtract a small fraction of the gradient from each weight.
6. **Repeat from step 2.** Each round, predictions get less terrible. After a few hundred rounds, the weights converge.

Let's walk through each step with real data and real numpy code.

In [ ]:
# 3 districts × 3 features — just enough to see the pattern
demo_feature_names = ['MedInc', 'HouseAge', 'AveRooms']
X_demo_raw = df[demo_feature_names].values[:3]
y_demo = y[:3]
X_demo = (X_demo_raw - X_demo_raw.mean(axis=0)) / X_demo_raw.std(axis=0)

print(f"X_demo shape: {X_demo.shape}  — 3 districts, 3 features each")
print(f"y_demo: {y_demo}  — actual house values\n")

# Step 1: Make up random weights — one per feature, plus a bias
np.random.seed(99)
weights_demo = np.random.randn(3) * 0.3
bias_demo = 0.0

print("weights =", weights_demo)
print("bias    =", bias_demo)
print(f"\nOne weight per feature: {list(zip(demo_feature_names, np.round(weights_demo, 3)))}")
print("These are random garbage. The model knows nothing yet.")

**Step 2: Multiply every row by the weights.** Each row gets dot-producted with the weight vector, and the bias is added. Let's expand one row to see what's happening inside the `@` operator:

In [ ]:
# Expand what happens for row 1 — the dot product
row = X_demo[0]
print("Row 1 features:", np.round(row, 3))
print("Weights:       ", np.round(weights_demo, 3))
print()

terms = []
total = 0
for j, name in enumerate(demo_feature_names):
    product = row[j] * weights_demo[j]
    total += product
    terms.append(f"({row[j]:+.2f} × {weights_demo[j]:+.3f}) = {product:+.4f}")
    print(f"  {name:10s}:  {row[j]:+.2f}  ×  {weights_demo[j]:+.3f}  =  {product:+.4f}")

total += bias_demo
print(f"  {'bias':10s}:  + {bias_demo:.3f}")
print(f"  {'prediction':10s}:  = {total:+.4f}")
print(f"\nThat's one dot product. The @ operator does this for all {len(X_demo)} rows at once:")
print()

# All rows at once
predictions_demo = X_demo @ weights_demo + bias_demo
print("predictions = X_demo @ weights_demo + bias_demo")
print(f"predictions = {np.round(predictions_demo, 4)}")
print(f"actual      = {np.round(y_demo, 4)}")

**Step 3: Measure how wrong we are.** Subtract actual from predicted, square the differences, take the mean. That's the MSE loss. Here's the full picture — weights multiplied with every row, producing predictions, then compared against actual values:

In [ ]:
# The diagram: weights × data rows → predictions → errors
draw_batch_gd_diagram(
    X_demo, y_demo, weights_demo, bias_demo, demo_feature_names, n_total_rows=len(X_train)
)

# Compute the loss
errors_demo = predictions_demo - y_demo
loss_before = (errors_demo ** 2).mean()

print(f"errors = predictions - actual = {np.round(errors_demo, 4)}")
print(f"MSE    = mean(errors²)        = {loss_before:.4f}")
print(f"\nWith random weights, we're way off. Now: which direction should we nudge each weight?")

**Step 4: Compute the gradients.** The gradient tells us, for each weight: "if I increase this weight slightly, does the loss go up or down, and by how much?"

The formula is `grad_w = (2/n) * (X.T @ errors)`. The key operation is `X.T @ errors` — it transposes the data matrix and multiplies by the error vector. This computes how much each feature contributed to the errors across all rows simultaneously.

In [ ]:
n = len(y_demo)
grad_w = (2/n) * (X_demo.T @ errors_demo)
grad_b = (2/n) * errors_demo.sum()

print("grad_w = (2/n) * (X.T @ errors)")
print(f"grad_w = {np.round(grad_w, 4)}")
print(f"grad_b = {grad_b:.4f}\n")

# What does each gradient mean?
for j, name in enumerate(demo_feature_names):
    direction = "decrease" if grad_w[j] > 0 else "increase"
    print(f"  w_{name:8s} gradient = {grad_w[j]:+.4f}  →  {direction} this weight to reduce loss")
print(f"  bias       gradient = {grad_b:+.4f}")

**Step 5: Nudge the weights.** Subtract a fraction of the gradient from each weight. The fraction is the learning rate — how big of a step we take. Then re-run the forward pass to verify the loss dropped.

In [ ]:
lr = 0.1
new_weights = weights_demo - lr * grad_w
new_bias = bias_demo - lr * grad_b

print(f"weights_new = weights - {lr} * grad_w\n")
for j, name in enumerate(demo_feature_names):
    print(f"  w_{name:8s}:  {weights_demo[j]:+.4f}  →  {new_weights[j]:+.4f}")
print(f"  bias      :  {bias_demo:+.4f}  →  {new_bias:+.4f}")

# Verify: re-run forward pass with updated weights
new_predictions = X_demo @ new_weights + new_bias
new_errors = new_predictions - y_demo
loss_after = (new_errors ** 2).mean()

print(f"\nMSE before: {loss_before:.4f}")
print(f"MSE after:  {loss_after:.4f}")
print(f"\nOne step, already better. Repeat this 200 times and the weights converge.")

### Batch vs. Stochastic Gradient Descent

What we just saw is **batch gradient descent**: compute the gradient using **every row**, then update once. The gradient is exact but can be slow with millions of rows.

| Variant | Rows per update | Gradient quality | Speed per step |
|---------|----------------|-----------------|----------------|
| **Batch GD** | All N rows | Exact | Slow |
| **Mini-batch SGD** | 32-256 rows | Good enough | Fast |
| **SGD** (pure) | 1 random row | Noisy | Very fast |

**Mini-batch SGD** is what frameworks like PyTorch use by default. Instead of computing `X.T @ errors` over all 16,000 rows, you grab a random batch of 64 rows, compute the gradient from just those, and update. The gradient is noisier but you get 250x more updates per pass through the data.

For our small dataset, batch GD works perfectly. For training image classifiers or language models on millions of examples, mini-batch SGD is the only practical option.

## Part 5: The Learning Rate — Step Size Matters!

The learning rate (α or lr) controls how big each step is:

- **Too small**: Tiny steps, takes forever to converge
- **Too large**: Overshoots and oscillates wildly
- **Just right**: Steady progress toward the minimum

Finding a good learning rate is often the first thing you tune.

In [ ]:
fig, axes = plot_learning_rate_comparison(
    f=simple_curve,
    df=derivative_of_curve,
    x_start=-2.5,
    learning_rates=[0.01, 0.1, 0.5],
    n_steps=15,
    x_range=(-3, 3)
)
plt.show()

print("LEFT (lr=0.01):  Too slow — barely moves after 15 steps")
print("MIDDLE (lr=0.1): Just right — smooth convergence to minimum")
print("RIGHT (lr=0.5):  Too fast — overshoots and bounces around")

### Explore It Yourself

The three plots above show hand-picked learning rates. What happens at 0.3? At 0.05? Use the sliders below to find the sweet spot — and watch what happens when you push it too far.

In [ ]:
from ipywidgets import FloatSlider, IntSlider, HBox, VBox, interactive_output
from IPython.display import display, Image
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import io

lr_slider = FloatSlider(min=0.01, max=0.95, step=0.01, value=0.1,
    description='Learning Rate:', style={'description_width': '120px'}, layout={'width': '500px'})
steps_slider = IntSlider(min=1, max=40, step=1, value=15,
    description='Steps:', style={'description_width': '120px'}, layout={'width': '500px'})

def explore_learning_rate(lr, n_steps):
    x_vals = np.linspace(-3, 3, 200)

    fig = Figure(figsize=(14, 5))
    FigureCanvasAgg(fig)
    ax_path, ax_loss = fig.subplots(1, 2)

    # Left: GD path on parabola
    ax_path.plot(x_vals, simple_curve(x_vals), 'b-', linewidth=2, alpha=0.5)
    x_current = -2.5
    path_x = [x_current]
    path_loss = [simple_curve(x_current)]
    diverged = False
    for _ in range(n_steps):
        gradient = derivative_of_curve(x_current)
        x_current = x_current - lr * gradient
        if abs(x_current) > 10:
            diverged = True
            break
        path_x.append(x_current)
        path_loss.append(simple_curve(x_current))

    final_loss = path_loss[-1]
    color = '#e74c3c' if diverged or final_loss > 1.0 else '#2ecc71'
    ax_path.plot(path_x, path_loss, 'o-', color=color, markersize=6, linewidth=1.5)
    ax_path.scatter([path_x[0]], [path_loss[0]], color='black', s=120, zorder=5, label='Start')
    ax_path.scatter([path_x[-1]], [path_loss[-1]], color=color, s=150, zorder=5, marker='*', label='End')
    ax_path.set_xlim(-3.5, 3.5)
    ax_path.set_ylim(-0.5, 10)
    ax_path.set_xlabel('x', fontsize=11)
    ax_path.set_ylabel('Loss = x²', fontsize=11)
    status = 'DIVERGED!' if diverged else f'Final loss: {final_loss:.4f}'
    ax_path.set_title(f'GD Path  (lr={lr:.2f}, {len(path_x)-1} steps)\n{status}',
                       fontsize=11, fontweight='bold', color=color)
    ax_path.legend(fontsize=9)
    ax_path.grid(True, alpha=0.3)

    # Right: Loss over steps
    ax_loss.plot(range(len(path_loss)), path_loss, 'o-', color=color, markersize=5, linewidth=2)
    ax_loss.set_xlabel('Step', fontsize=11)
    ax_loss.set_ylabel('Loss', fontsize=11)
    ax_loss.set_title('Loss Over Steps', fontsize=11, fontweight='bold')
    ax_loss.grid(True, alpha=0.3)
    ax_loss.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    display(Image(data=buf.getvalue()))

out = interactive_output(explore_learning_rate, {'lr': lr_slider, 'n_steps': steps_slider})
display(VBox([HBox([lr_slider, steps_slider]), out]))

## Part 6: Feature Scaling — Why We Normalized the Data

In Part 2, we quietly normalized the income data before computing the loss. Now that we understand gradient descent and learning rates, we can see why that step was essential.

The raw income feature ranges from 0.5 to 15. The gradient formula multiplies errors by the feature values, so the weight gradient ends up roughly 10x larger than the bias gradient. A learning rate that works for one is wrong for the other — and this distorts the entire loss landscape that GD has to navigate:

In [ ]:
# How feature scaling changes the loss landscape
plot_scaling_landscape(X_train, X_train_norm, y_train, predict, mse_loss)

print("Left:  Unscaled — the valley is stretched. GD crawls along the narrow direction.")
print("Right: Scaled — the valley is round. GD heads straight to the minimum.")
print(f"\n150 steps (lr=0.001) vs 20 steps (lr=0.1) to reach the same quality.")

### Standardization in Code

The formula is straightforward:

$$x_{\text{scaled}} = \frac{x - \text{mean}}{\text{std}}$$

We did this manually in Part 2 with numpy. In practice, scikit-learn's `StandardScaler` does the same math — but it also **remembers** the training statistics so you can apply the exact same transformation to test data or new inputs later.

In [ ]:
# Manual standardization (what we wrote in Part 2)
print("Manual numpy approach:")
print(f"  X_train_norm = (X_train - X_train.mean()) / X_train.std()")
print(f"  Result — mean: {X_train_norm.mean():.6f}, std: {X_train_norm.std():.4f}")

# The scikit-learn equivalent
from sklearn.preprocessing import StandardScaler

scaler_demo = StandardScaler()
X_scaler_result = scaler_demo.fit_transform(X_train.reshape(-1, 1)).flatten()

print(f"\nStandardScaler approach:")
print(f"  scaler = StandardScaler()")
print(f"  X_scaled = scaler.fit_transform(X_train)")
print(f"  Result — mean: {X_scaler_result.mean():.6f}, std: {X_scaler_result.std():.4f}")

print(f"\nIdentical results: {np.allclose(X_train_norm, X_scaler_result)}")
print(f"\nStandardScaler stores the training statistics:")
print(f"  scaler.mean_  = {scaler_demo.mean_[0]:.4f}")
print(f"  scaler.scale_ = {scaler_demo.scale_[0]:.4f}")
print(f"\nFor test data: scaler.transform(X_test) — uses training mean/std, not test's.")

### The Impact on Gradient Descent

Standardization is simple math. But does it actually matter in practice? Let's run the same gradient descent algorithm three ways and compare convergence:

In [ ]:
# Run gradient descent on raw vs normalized features — same algorithm, same initial params
configs = [
    ('Scaled (lr=0.1)', X_train_norm, 0.1, '#2ecc71', '-'),
    ('Unscaled (lr=0.001)', X_train, 0.001, '#e67e22', '--'),
    ('Unscaled (lr=0.1)', X_train, 0.1, '#e74c3c', ':'),
]
plot_scaling_convergence(configs, y_train, predict, mse_loss)

print("Green:  Scaled data + good lr     → converges in ~20 iterations")
print("Orange: Unscaled data + tiny lr   → converges, but 100x slower")
print("Red:    Unscaled data + same lr   → DIVERGES — gradients explode")
print("\nWithout scaling, you either crawl or crash. There's no good learning rate.")

## Part 7: Training on Real Data

We explored the loss landscape with sliders. Now let's let gradient descent find the optimal weight and bias automatically.

For linear regression with MSE loss, the gradients are:

$$\frac{\partial \text{Loss}}{\partial w} = \frac{2}{n} \sum (\text{prediction} - \text{actual}) \cdot x$$

$$\frac{\partial \text{Loss}}{\partial b} = \frac{2}{n} \sum (\text{prediction} - \text{actual})$$

These tell us how to adjust each parameter to reduce the loss.

In [ ]:
def compute_gradients(X, y, weight, bias):
    """Compute gradients for weight and bias"""
    predictions = predict(X, weight, bias)
    errors = predictions - y
    n = len(y)
    
    gradient_weight = (2 / n) * (errors * X).sum()
    gradient_bias = (2 / n) * errors.sum()
    
    return gradient_weight, gradient_bias

# Initialize parameters
weight = 0.0
bias = 0.0
learning_rate = 0.1
n_iterations = 200

# Track history for visualization
loss_history = []
weight_history = [weight]
bias_history = [bias]

print("Training on California Housing data...")
print("=" * 60)

for i in range(n_iterations):
    # Forward pass: make predictions
    predictions = predict(X_train_norm, weight, bias)
    loss = mse_loss(predictions, y_train)
    loss_history.append(loss)
    
    # Compute gradients
    grad_w, grad_b = compute_gradients(X_train_norm, y_train, weight, bias)
    
    # Update parameters
    weight = weight - learning_rate * grad_w
    bias = bias - learning_rate * grad_b
    
    weight_history.append(weight)
    bias_history.append(bias)
    
    # Print progress
    if i < 5 or i % 50 == 0 or i == n_iterations - 1:
        print(f"Iter {i:3d}: Loss = {loss:.4f}, w = {weight:.4f}, b = {bias:.4f}")

print("\nTraining complete!")

In [ ]:
# Visualize the training progress
plot_training_progress(loss_history, weight_history, bias_history)

print("All parameters stabilize as we approach the minimum loss.")

### Experiment with Training

How does the learning rate affect real data training? What if we stop too early? Slide the values and watch the loss curve and fitted line change in real time.

In [ ]:
train_lr_slider = FloatSlider(min=0.01, max=1.0, step=0.01, value=0.1,
    description='Learning Rate:', style={'description_width': '120px'}, layout={'width': '450px'})
train_iters_slider = IntSlider(min=5, max=500, step=5, value=200,
    description='Iterations:', style={'description_width': '120px'}, layout={'width': '450px'})

def explore_training(lr, n_iters):
    w_train, b_train = 0.0, 0.0
    n = len(X_train_norm)
    losses = []

    for iteration in range(n_iters):
        current_predictions = predict(X_train_norm, w_train, b_train)
        errors = current_predictions - y_train
        w_train -= lr * (2/n) * (errors * X_train_norm).sum()
        b_train -= lr * (2/n) * errors.sum()
        losses.append(mse_loss(current_predictions, y_train))
        if losses[-1] > 1e6:
            break

    fig = Figure(figsize=(14, 5))
    FigureCanvasAgg(fig)
    ax_loss, ax_fit = fig.subplots(1, 2)

    # Left: Loss curve
    if max(losses) < 1e6:
        ax_loss.plot(losses, 'b-', linewidth=2)
        ax_loss.set_ylabel('MSE Loss', fontsize=11)
        if len(losses) > 1:
            ax_loss.set_ylim(0, max(losses[0] * 1.1, losses[-1] * 2))
    else:
        ax_loss.text(0.5, 0.5, 'DIVERGED!\nLearning rate too high',
                     transform=ax_loss.transAxes, ha='center', va='center',
                     fontsize=16, color='red', fontweight='bold')
    ax_loss.set_xlabel('Iteration', fontsize=11)
    ax_loss.set_title(f'Training Loss  (lr={lr:.2f}, {n_iters} iters)', fontsize=11, fontweight='bold')
    ax_loss.grid(True, alpha=0.3)

    # Right: Fitted line
    sample_rng = np.random.RandomState(42)
    sample_idx = sample_rng.choice(len(X_train), 800, replace=False)
    ax_fit.scatter(X_train[sample_idx], y_train[sample_idx], alpha=0.3, s=15, color='steelblue')
    x_line = np.linspace(X_train.min(), X_train.max(), 100)
    y_line = predict((x_line - X_mean) / X_std, w_train, b_train)
    ax_fit.plot(x_line, y_line, 'r-', linewidth=2.5)
    ax_fit.set_xlabel('Median Income ($10,000s)', fontsize=11)
    ax_fit.set_ylabel('House Value ($100,000s)', fontsize=11)
    final_loss = losses[-1] if losses[-1] < 1e6 else float('inf')
    ax_fit.set_title(f'w={w_train:.3f}, b={b_train:.3f}, MSE={final_loss:.4f}',
                     fontsize=11, fontweight='bold')
    ax_fit.set_ylim(-0.5, 5.5)
    ax_fit.grid(True, alpha=0.3)

    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    display(Image(data=buf.getvalue()))

out = interactive_output(explore_training, {'lr': train_lr_slider, 'n_iters': train_iters_slider})
display(VBox([HBox([train_lr_slider, train_iters_slider]), out]))

In [ ]:
# Visualize the learned model on the data
plot_model_fit(X_train, y_train, X_mean, X_std, weight, bias, predict)

print(f"Final model (on normalized inputs): y = {weight:.3f} × x_norm + {bias:.3f}")
print(f"Where x_norm = (income - {X_mean:.2f}) / {X_std:.2f}")

### The Practical Shortcut

We wrote gradient descent from scratch to understand the mechanics. For linear regression specifically, scikit-learn can solve it in one step — no iterations, no learning rate to tune:

In [ ]:
# Everything we just did in 200 iterations, solved in 1 line
from sklearn.linear_model import LinearRegression

sklearn_lr = LinearRegression()
sklearn_lr.fit(X_train_norm.reshape(-1, 1), y_train)

print("scikit-learn LinearRegression (closed-form solution):")
print(f"  Weight: {sklearn_lr.coef_[0]:.4f}  (our gradient descent: {weight:.4f})")
print(f"  Bias:   {sklearn_lr.intercept_:.4f}  (our gradient descent: {bias:.4f})")
print(f"\nIdentical answers. sklearn uses the 'normal equation' — direct linear algebra.")
print(f"So why learn gradient descent? Because neural networks have no closed-form")
print(f"solution. Gradient descent is the only way to train them.")

### The Training Loop — A Universal Pattern

What we just wrote is THE core training loop that powers all of machine learning:

```python
for each iteration:
    predictions = model(inputs)               # Forward pass
    loss = loss_function(predictions, targets)
    gradients = compute_gradients()           # Backward pass
    parameters -= learning_rate * gradients   # Update step
```

This same pattern — with more complex models and automatic gradient computation — trains image classifiers, language models, speech recognition, and everything else in deep learning. The model changes. The training loop doesn't.

**A note on terminology:** We ran 200 "iterations" above. Each iteration used all 16,512 training rows (batch GD), so each iteration is also one **epoch** — one complete pass through the data. Remember the `epoch` column from Lesson 2's `fine_tune()` output? Same thing. With mini-batch SGD (which PyTorch uses), one epoch takes many iterations because you process 64 rows at a time — you need ~250 mini-batches to get through 16k rows once.

## Part 8: From Scratch to PyTorch

We coded gradients by hand to understand the mechanics. In practice, frameworks like **PyTorch** compute gradients automatically via **automatic differentiation** — no manual calculus required.

Let's verify our hand-coded implementation matches PyTorch on the same data:

In [ ]:
import torch
import torch.nn as nn

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_norm).reshape(-1, 1)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1)

# Create a simple linear model
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Train for same number of iterations
pytorch_losses = []
for i in range(200):
    # Forward pass
    predictions = model(X_train_tensor)
    loss = loss_fn(predictions, y_train_tensor)
    pytorch_losses.append(loss.item())
    
    # Backward pass (automatic!)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Comparison: Our Implementation vs PyTorch")
print("=" * 50)
print(f"Our weight:     {weight:.4f}")
print(f"PyTorch weight: {model.weight.item():.4f}")
print(f"\nOur bias:       {bias:.4f}")
print(f"PyTorch bias:   {model.bias.item():.4f}")
print("\n✓ Same result! PyTorch just automates the gradient computation.")

## Part 9: Evaluation — How Good is Our Model?

Whether we train from scratch or use PyTorch, we converge to the same parameters. But matching weights doesn't tell us if the model is actually useful. How well does it predict house values it's never seen?

Different metrics answer this from different angles:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MSE** | mean((pred - actual)²) | Average squared error |
| **RMSE** | √MSE | Same units as target |
| **MAE** | mean(\|pred - actual\|) | Average absolute error |
| **R²** | 1 - SS_res/SS_tot | Proportion of variance explained |

R² is especially intuitive: R²=0.5 means the model explains 50% of the variance in house values.

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute all regression metrics"""
    errors = y_pred - y_true
    
    mse = (errors ** 2).mean()
    rmse = np.sqrt(mse)
    mae = np.abs(errors).mean()
    
    ss_residual = (errors ** 2).sum()
    ss_total = ((y_true - y_true.mean()) ** 2).sum()
    r2 = 1 - (ss_residual / ss_total)
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R²': r2}

# Evaluate on both train and test sets
train_predictions = predict(X_train_norm, weight, bias)
test_predictions = predict(X_test_norm, weight, bias)

train_metrics = compute_metrics(y_train, train_predictions)
test_metrics = compute_metrics(y_test, test_predictions)

print("Model Evaluation")
print("=" * 50)
print(f"{'Metric':<10} {'Train':>15} {'Test':>15}")
print("-" * 50)
for metric_name in ['MSE', 'RMSE', 'MAE', 'R²']:
    print(f"{metric_name:<10} {train_metrics[metric_name]:>15.4f} {test_metrics[metric_name]:>15.4f}")
print("=" * 50)

print(f"\n✓ R² = {test_metrics['R²']:.2f} means income explains {test_metrics['R²']*100:.0f}% of house value variance")
print("  The remaining variance comes from other factors (location, size, etc.)")

In [ ]:
# In practice, use scikit-learn's built-in metrics:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Same metrics via scikit-learn:")
print(f"  MSE:  {mean_squared_error(y_test, test_predictions):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, test_predictions)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test, test_predictions):.4f}")
print(f"  R²:   {r2_score(y_test, test_predictions):.4f}")
print(f"\nMatches our hand-coded compute_metrics(). Understanding the math matters,")
print(f"but in production you'd use these one-liners.")

### Making Predictions

Metrics tell us the model works. Let's actually use it — predict house values for districts the model has never seen.

In [ ]:
# Predict house values for districts with different income levels
new_incomes = np.array([2, 4, 6, 8, 10, 12])

# Remember to normalize using the TRAINING statistics!
new_incomes_normalized = (new_incomes - X_mean) / X_std
predicted_values = predict(new_incomes_normalized, weight, bias)

print("Predictions for New Districts")
print("=" * 55)
print(f"{'Median Income':>20} {'Predicted Value':>20}")
print("-" * 55)
for income, value in zip(new_incomes, predicted_values):
    # Convert from $10,000s and $100,000s to readable format
    print(f"${income*10:>16,}k     ${value*100:>16,.0f}k")
print("=" * 55)

print("\nNote: Always normalize new inputs using TRAINING set statistics!")

## Part 10: Overfitting and Underfitting — The Central Challenge

Our model explains 46% of the variance (R² = 0.46 from Part 9). Not bad for one feature, but not great either. You might think: just make the model more complex! Give it more parameters!

That instinct is dangerous. When a model has **too many parameters relative to the data**, it starts memorizing noise instead of learning the real pattern. This is called **overfitting**. Its opposite — a model too simple to capture the real pattern — is **underfitting**.

Polynomial regression lets us control complexity directly: a degree-$d$ polynomial has $d+1$ parameters. Let's crank it up and watch what happens.

In [ ]:
# 15 points with a clear quadratic pattern + noise
np.random.seed(42)
x_small = np.linspace(0.5, 7.5, 15)
y_small = 2 + 0.8 * x_small - 0.05 * x_small**2 + np.random.normal(0, 0.4, 15)

plot_polynomial_fits(
    x_small, y_small, degrees=[1, 3, 12],
    titles=['Degree 1: Underfitting', 'Degree 3: Good fit', 'Degree 12: Overfitting'],
    colors=['red', 'green', 'purple']
)

print("Degree 1:  Too simple — can't capture the curve (UNDERFITTING)")
print("Degree 3:  Captures the trend without chasing noise")
print("Degree 12: Passes through every point — memorized the training data (OVERFITTING)")
print(f"\n15 data points, 13 parameters (degree 12) → almost as many parameters as data points!")

### Explore Overfitting Live

Slide the polynomial degree up from 1 to 14 and watch the model evolve. At what degree does it start chasing individual data points instead of learning the overall pattern?

In [ ]:
degree_slider = IntSlider(min=1, max=14, step=1, value=1,
    description='Poly Degree:', style={'description_width': '120px'}, layout={'width': '500px'})

def explore_polynomial(degree):
    from sklearn.preprocessing import PolynomialFeatures as PF

    poly = PF(degree=degree, include_bias=False)
    X_poly_train = poly.fit_transform(x_small.reshape(-1, 1))
    model = LinearRegression().fit(X_poly_train, y_small)

    train_mse = mse_loss(model.predict(X_poly_train), y_small)

    fig = Figure(figsize=(14, 5))
    FigureCanvasAgg(fig)
    ax_fit, ax_info = fig.subplots(1, 2)

    # Left: Polynomial fit on the 15 points
    ax_fit.scatter(x_small, y_small, color='black', s=80, zorder=5, label='Training data (15 pts)')
    x_plot = np.linspace(-0.5, 8.5, 300)
    y_true = 2 + 0.8 * x_plot - 0.05 * x_plot**2
    ax_fit.plot(x_plot, y_true, 'gray', linewidth=1, linestyle='--', alpha=0.5, label='True pattern')
    y_pred = model.predict(poly.transform(x_plot.reshape(-1, 1)))
    y_pred_clipped = np.clip(y_pred, -2, 8)
    color = '#2ecc71' if degree <= 5 else '#e67e22' if degree <= 8 else '#e74c3c'
    ax_fit.plot(x_plot, y_pred_clipped, color=color, linewidth=2.5,
                label=f'Degree {degree} ({degree+1} params)')
    ax_fit.set_ylim(-2, 8)
    ax_fit.set_xlabel('x', fontsize=11)
    ax_fit.set_ylabel('y', fontsize=11)
    ax_fit.set_title(f'Polynomial Degree {degree}', fontsize=12, fontweight='bold', color=color)
    ax_fit.legend(fontsize=9)
    ax_fit.grid(True, alpha=0.3)

    # Right: Parameters vs data points ratio + train MSE
    ratio = (degree + 1) / 15
    bar_colors = ['#3498db', '#e74c3c' if ratio > 0.6 else '#2ecc71']
    ax_info.bar(['Train MSE', 'Params / Data'], [train_mse, ratio], color=bar_colors, width=0.5)
    ax_info.text(0, train_mse + 0.03, f'{train_mse:.4f}', ha='center', fontsize=12, fontweight='bold')
    ax_info.text(1, ratio + 0.03, f'{degree+1}/15 = {ratio:.0%}', ha='center', fontsize=12, fontweight='bold')

    if ratio > 0.8:
        verdict = 'Almost as many parameters as data points!'
        verdict_color = '#e74c3c'
    elif ratio > 0.4:
        verdict = 'Getting complex — watch for overfitting'
        verdict_color = '#e67e22'
    elif train_mse > 0.5:
        verdict = 'Too simple — underfitting'
        verdict_color = '#e67e22'
    else:
        verdict = 'Good balance of fit and simplicity'
        verdict_color = '#2ecc71'

    ax_info.set_title(verdict, fontsize=11, fontweight='bold', color=verdict_color)
    ax_info.set_ylim(0, max(1.2, train_mse * 1.3))
    ax_info.grid(True, alpha=0.3, axis='y')

    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    display(Image(data=buf.getvalue()))

out = interactive_output(explore_polynomial, {'degree': degree_slider})
display(VBox([degree_slider, out]))

### Detecting Overfitting: The Train/Test Gap

The polynomial visual made overfitting obvious — we could *see* the wiggly line chasing noise. But in real ML you're working with dozens of features and thousands of dimensions. You can't eyeball the fit.

Instead, you track **train error vs test error** as you increase model complexity. When test error starts rising while train error keeps falling, that gap IS overfitting. This is the standard diagnostic.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Deliberately tiny sample from California Housing — forces overfitting at high degrees
np.random.seed(42)
train_idx = np.random.choice(len(X_train_norm), 50, replace=False)
test_idx = np.random.choice(len(X_test_norm), 200, replace=False)

X_small_train, y_small_train = X_train_norm[train_idx], y_train[train_idx]
X_small_test, y_small_test = X_test_norm[test_idx], y_test[test_idx]

degrees = list(range(1, 16))
train_mses, test_mses = [], []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_small_train.reshape(-1, 1))
    X_test_poly = poly.transform(X_small_test.reshape(-1, 1))
    poly_model = LinearRegression().fit(X_train_poly, y_small_train)
    train_mses.append(mse_loss(poly_model.predict(X_train_poly), y_small_train))
    test_mses.append(mse_loss(poly_model.predict(X_test_poly), y_small_test))

best_degree = plot_complexity_curve(degrees, train_mses, test_mses)

print(f"50 training points, sweeping polynomial degrees 1–15:")
print(f"  Train MSE drops monotonically — more parameters always fit training data better")
print(f"  Test MSE hits a minimum at degree {best_degree}, then explodes")
print(f"\nThis divergence is how you detect overfitting when you can't visualize the fit.")

### Extrapolation: Where Overfitting Gets Dangerous

Overfit models don't just fail on held-out test data — they **completely collapse** when asked to predict outside the training range.

In [ ]:
# Degree 3 vs 12: what happens outside the training range?
plot_extrapolation(x_small, y_small)

print("Inside the training range (white): both models look reasonable")
print("Outside the training range (red): degree 12 shoots off wildly")
print("\nThis is why overfit models are dangerous in production!")

### Overfitting Summary

| Symptom | What It Means |
|---------|---------------|
| Training loss keeps dropping | Model memorizes training data |
| Test loss starts rising | Model fails to generalize |
| Wild predictions outside training range | Learned noise, not the real pattern |
| Train/test gap widens with complexity | Too many parameters for the data |

The recipe for overfitting: **too many parameters + too little data + too much noise**. This is the exact same phenomenon in neural networks — just with millions of parameters instead of 12.

### Early Stopping — Catching It During Training

We detected overfitting above by comparing different polynomial degrees after training. In practice, you want to catch it **during** training by monitoring test loss at each iteration.

Let's reuse our 15 synthetic points — split 10 for training, 5 for testing — and train a degree-8 polynomial with gradient descent. With 8 features and only 10 training points, the model will overfit as it trains:

In [ ]:
# Split the 15 synthetic points: 10 train, 5 test
x_es_train, y_es_train = x_small[:10], y_small[:10]
x_es_test, y_es_test = x_small[10:], y_small[10:]

# Create degree-8 polynomial features (8 features from 10 points = recipe for overfitting)
poly_es = PolynomialFeatures(degree=8, include_bias=False)
X_es_train = poly_es.fit_transform(x_es_train.reshape(-1, 1))
X_es_test = poly_es.transform(x_es_test.reshape(-1, 1))

# Standardize (polynomial features blow up without this)
poly_mean = X_es_train.mean(axis=0)
poly_std = X_es_train.std(axis=0) + 1e-8
X_es_train_s = (X_es_train - poly_mean) / poly_std
X_es_test_s = (X_es_test - poly_mean) / poly_std

# Gradient descent — track both train and test loss
weights_es = np.zeros(8)
bias_es = 0.0
lr_es = 0.01
train_losses_es, test_losses_es = [], []
best_test_loss = float('inf')
best_iter = 0

for i in range(500):
    train_preds = X_es_train_s @ weights_es + bias_es
    test_preds = X_es_test_s @ weights_es + bias_es
    
    train_loss = ((train_preds - y_es_train) ** 2).mean()
    test_loss = ((test_preds - y_es_test) ** 2).mean()
    train_losses_es.append(train_loss)
    test_losses_es.append(test_loss)
    
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_iter = i
    
    # Update using training data only
    errors = train_preds - y_es_train
    n = len(y_es_train)
    weights_es -= lr_es * (2/n) * (X_es_train_s.T @ errors)
    bias_es -= lr_es * (2/n) * errors.sum()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(train_losses_es, 'b-', linewidth=2, label='Training Loss')
axes[0].plot(test_losses_es, 'orange', linewidth=2, linestyle='--', label='Test Loss')
axes[0].axvline(x=best_iter, color='red', linestyle=':', linewidth=2, alpha=0.7)
axes[0].annotate(f'Best test loss\n(iter {best_iter})', xy=(best_iter, best_test_loss),
                 xytext=(best_iter + 100, max(test_losses_es) * 0.5),
                 fontsize=11, fontweight='bold', color='red',
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))
axes[0].fill_between(range(best_iter, 500),
                     train_losses_es[best_iter:],
                     test_losses_es[best_iter:],
                     alpha=0.15, color='red', label='Overfitting gap')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Degree-8 Polynomial on 10 Points — Real Overfitting', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: zoomed view around the divergence point
zoom_start = max(0, best_iter - 20)
zoom_end = min(500, best_iter + 150)
axes[1].plot(range(zoom_start, zoom_end), train_losses_es[zoom_start:zoom_end],
             'b-', linewidth=2, label='Training Loss')
axes[1].plot(range(zoom_start, zoom_end), test_losses_es[zoom_start:zoom_end],
             'orange', linewidth=2, linestyle='--', label='Test Loss')
axes[1].axvline(x=best_iter, color='red', linestyle=':', linewidth=2, alpha=0.7,
                label=f'Stop here (iter {best_iter})')
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('MSE Loss', fontsize=12)
axes[1].set_title('Zoomed In — The Moment It Diverges', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Training loss keeps dropping — the model memorizes the 10 training points.")
print(f"Test loss hits minimum at iteration {best_iter}, then rises as the model overfits.")
print(f"\nEarly stopping = save the best weights (at iter {best_iter}) and stop.")
print(f"We'll implement this properly with neural networks in the next lesson.")

## Part 11: Multiple Features — Using All Available Information

We've been working with a single feature (income) and hit R² = 0.46. We've also learned to watch for overfitting when adding complexity. Now let's add complexity the right way — not by fitting higher-degree polynomials, but by giving the model **more information**.

We have 8 features. Each one adds signal the model can use. With one feature, the model was:
```
prediction = weight × income + bias
```

With multiple features, it becomes a **matrix multiplication**:

$$\text{prediction} = w_1 \cdot x_1 + w_2 \cdot x_2 + \ldots + w_8 \cdot x_8 + b$$

Or in matrix form: `predictions = X @ weights + bias`

Here's what that looks like concretely for 3 districts and 3 features:

```
 [income₁  age₁  rooms₁]   [w_income]   [bias]   [pred₁]
 [income₂  age₂  rooms₂] × [w_age   ] + [bias] = [pred₂]
 [income₃  age₃  rooms₃]   [w_rooms ]   [bias]   [pred₃]
```

Each row is a district. Each column is a feature. The model learns one weight per feature.

In [ ]:
# Progressive feature addition: watch R² climb as we add relevant features
feature_sets = [
    (['MedInc'], 'Income only'),
    (['MedInc', 'Latitude', 'Longitude'], 'Income + Location'),
    (['MedInc', 'Latitude', 'Longitude', 'HouseAge', 'AveRooms'], '+ Housing characteristics'),
    (list(california.feature_names), 'All 8 features'),
]

r2_scores = []

print("Adding features progressively:")
print("=" * 60)

for feature_list, label in feature_sets:
    X_subset = df[feature_list].values
    X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(X_subset, y, test_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sub)
    X_test_scaled = scaler.transform(X_test_sub)
    
    linear_model = LinearRegression().fit(X_train_scaled, y_train_sub)
    r2 = compute_metrics(y_test_sub, linear_model.predict(X_test_scaled))['R²']
    r2_scores.append(r2)
    
    print(f"  {len(feature_list)} features ({label:30s}): R² = {r2:.3f}")

labels = [fs[1] for fs in feature_sets]
plot_r2_progression(labels, r2_scores)

print(f"\nLocation (lat/long) added +{(r2_scores[1]-r2_scores[0])*100:.1f}% — geography matters a lot for housing!")

### Feature Importance: Which Features Matter Most?

When all features are scaled to the same range (standardized), the **magnitude of each weight** tells us how important that feature is. Larger weight = stronger influence on the prediction.

In [ ]:
# Train on all 8 features and visualize feature importance
X_all = df[california.feature_names].values
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

scaler_all = StandardScaler()
X_train_all_scaled = scaler_all.fit_transform(X_train_all)
X_test_all_scaled = scaler_all.transform(X_test_all)

full_model = LinearRegression().fit(X_train_all_scaled, y_train_all)
feature_weights = full_model.coef_

plot_feature_bars(california.feature_names, feature_weights,
                  title='Feature Importance: What Drives House Prices?')

print("Interpreting the top features:")
print(f"  MedInc  (+{feature_weights[0]:.2f}): Higher income → higher house value")
print(f"  Latitude ({feature_weights[6]:.2f}): Further north → lower value (LA/SF are south)")
print(f"  Longitude ({feature_weights[7]:.2f}): Further west → higher value (coast is west)")

### What Happens When You Add Irrelevant Features?

Not all features help. What if we add random noise columns — features with zero real relationship to house prices?

This connects back to overfitting: irrelevant features give the model more "knobs to turn" without adding real signal. With enough noise features or too little data, the model starts fitting the noise.

In [ ]:
# Add 5 pure noise features (random numbers, no relationship to house prices)
np.random.seed(42)
noise_features = np.random.randn(len(X_all), 5)
X_with_noise = np.hstack([X_all, noise_features])

noise_names = ['noise_1', 'noise_2', 'noise_3', 'noise_4', 'noise_5']
all_names = list(california.feature_names) + noise_names

# Train/test split
X_train_noise, X_test_noise, y_train_noise, y_test_noise = train_test_split(
    X_with_noise, y, test_size=0.2, random_state=42
)

scaler_noise = StandardScaler()
X_train_noise_scaled = scaler_noise.fit_transform(X_train_noise)
X_test_noise_scaled = scaler_noise.transform(X_test_noise)

noise_model = LinearRegression().fit(X_train_noise_scaled, y_train_noise)
r2_with_noise = compute_metrics(y_test_noise, noise_model.predict(X_test_noise_scaled))['R²']
r2_without_noise = r2_scores[-1]

plot_feature_bars(all_names, noise_model.coef_,
                  title='Real Features vs Noise Features', noise_prefix='noise')

print(f"R² without noise features: {r2_without_noise:.3f}")
print(f"R² with 5 noise features:  {r2_with_noise:.3f}")
print(f"\nNoise features got near-zero weights — the model mostly ignores them.")
print("But on small datasets, noise features can seriously hurt performance.")

### Multiple Features Summary

| Features Used | R² (Test) | Insight |
|--------------|-----------|---------|
| Income only | ~0.46 | One feature explains 46% of variance |
| + Location | ~0.57 | Geography matters a lot for housing |
| + Housing characteristics | ~0.58 | Small additional gain |
| All 8 features | ~0.58 | Extra features don't always help! |
| All 8 + 5 noise features | ~0.58 | Noise features mostly ignored |

Key takeaways:
- More **relevant** features improve predictions (income + location = big jump)
- Adding irrelevant or redundant features can slightly **hurt** — not all features are helpful
- The matrix form `y = X @ w + b` scales to any number of features
- Feature importance (weight magnitude on scaled data) reveals what matters

## Part 12: The Limits of Linear Regression

We've spent this entire notebook on linear models. Time for an honest question: **how far can a straight line actually take you?**

Let's find out with a different dataset: fuel efficiency of cars from the 1970s and 80s. This was the era of V8 muscle cars colliding with the oil crisis — manufacturers suddenly had to care about miles per gallon. Given a car's specs, can we predict its fuel efficiency?

In [ ]:
# Load the Auto MPG dataset
col_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
             'acceleration', 'model_year', 'origin', 'car_name']
mpg_df = pd.read_csv(DATA_PATH / "auto_mpg" / "auto-mpg.data",
                      sep=r'\s+', names=col_names, na_values='?').dropna()

# The extremes: gas guzzlers vs sippers
guzzlers = mpg_df.nsmallest(3, 'mpg')[['car_name', 'cylinders', 'weight', 'horsepower', 'mpg']]
sippers = mpg_df.nlargest(3, 'mpg')[['car_name', 'cylinders', 'weight', 'horsepower', 'mpg']]

print(f"Auto MPG: {len(mpg_df)} cars from the 1970s-80s\n")
print("Gas guzzlers:")
print(guzzlers.to_string(index=False))
print("\nFuel sippers:")
print(sippers.to_string(index=False))

# Fit linear regression on 6 numeric features
feature_cols = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X_mpg = mpg_df[feature_cols].values
y_mpg = mpg_df['mpg'].values

X_mpg_train, X_mpg_test, y_mpg_train, y_mpg_test = train_test_split(
    X_mpg, y_mpg, test_size=0.2, random_state=42)
scaler_mpg = StandardScaler()
mpg_linear = LinearRegression().fit(scaler_mpg.fit_transform(X_mpg_train), y_mpg_train)
mpg_preds = mpg_linear.predict(scaler_mpg.transform(X_mpg_test))
mpg_r2 = compute_metrics(y_mpg_test, mpg_preds)['R²']

# The curve a line can't capture
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(mpg_df['weight'], mpg_df['mpg'], alpha=0.4, s=30, color='steelblue')
weight_only_model = LinearRegression().fit(mpg_df[['weight']].values, mpg_df['mpg'].values)
x_line = np.linspace(mpg_df['weight'].min(), mpg_df['weight'].max(), 100)
ax.plot(x_line, weight_only_model.predict(x_line.reshape(-1, 1)), 'r-', linewidth=2.5, label='Linear fit')
ax.set_xlabel('Vehicle Weight (lbs)', fontsize=12)
ax.set_ylabel('Miles Per Gallon', fontsize=12)
ax.set_title("Weight vs MPG: The Curve a Line Can't Capture", fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nLinear regression R² = {mpg_r2:.2f} (using all 6 features)")
print("Decent — but look at the scatter. The real relationship curves.")
print("No matter how many features we add, a weighted sum can't bend.")

### Can We Do Better?

In Part 8 we introduced PyTorch. Let's use it to build something more powerful: a **neural network** with one hidden layer. Same training loop, same loss function (MSE), same optimizer (gradient descent). The only new ingredient is a non-linear activation function between layers — that's what lets the model learn curves.

In [ ]:
# Same data, same gradient descent — but a NON-LINEAR model
X_train_tensor = torch.FloatTensor(scaler_mpg.transform(X_mpg_train))
X_test_tensor = torch.FloatTensor(scaler_mpg.transform(X_mpg_test))
y_train_tensor = torch.FloatTensor(y_mpg_train).reshape(-1, 1)

# 2-layer neural network: Linear → ReLU → Linear
mpg_nn = nn.Sequential(
    nn.Linear(6, 16),   # 6 features → 16 hidden neurons
    nn.ReLU(),           # Non-linear activation (this is the key!)
    nn.Linear(16, 1)     # 16 hidden → 1 output (MPG prediction)
)

optimizer = torch.optim.SGD(mpg_nn.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# Same training loop from Part 7
for epoch in range(1000):
    preds = mpg_nn(X_train_tensor)
    loss = loss_fn(preds, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Evaluate
with torch.no_grad():
    nn_preds = mpg_nn(X_test_tensor).numpy().flatten()
nn_r2 = compute_metrics(y_mpg_test, nn_preds)['R²']

# Side-by-side: Linear vs Neural Network
plot_actual_vs_predicted(
    y_mpg_test,
    [mpg_preds, nn_preds],
    ['Linear Regression', 'Neural Network (1 hidden layer)'],
    [mpg_r2, nn_r2],
    ['coral', '#2ecc71']
)

print(f"Linear regression:  R² = {mpg_r2:.2f}")
print(f"Neural network:     R² = {nn_r2:.2f}")
print(f"\nSame data. Same loss function. Same gradient descent.")
print(f"The only difference: one hidden layer with a non-linear activation (ReLU).")

That's the punchline of this notebook. Everything we learned — loss functions, gradients, gradient descent, learning rates, overfitting — applies directly to neural networks. The training loop is identical. We just swap in a model that can bend.

In the next lesson, we'll open up that neural network and understand exactly how `Linear → ReLU → Linear` lets the model learn curves.

## Key Takeaways

1. **Regression** predicts a continuous number (house price, fuel efficiency). The other pillar of supervised learning alongside classification.
2. **Loss function** (MSE) measures how wrong the model is. Training = minimizing loss. Lower is better.
3. **Gradient descent** minimizes loss: compute the **gradient** (slope of the loss), take a step opposite to it. Repeat.
4. **Learning rate** controls step size — too small is slow, too large overshoots and diverges.
5. **Feature scaling** (standardization) is critical. Without it, gradient descent takes lopsided steps and may fail to converge.
6. **Train/test split** is non-negotiable. Always evaluate on data the model never saw during training.
7. **Overfitting** = model memorizes training noise instead of learning the real pattern. Detect it by watching the gap between train loss and test loss.
8. **Linear models have a ceiling** — they combine features with a weighted sum, so they can never learn curved relationships. Neural networks can.

## What's Next?

We've mastered the mechanics: loss functions, gradients, gradient descent, overfitting. These concepts don't change — they're the foundation of all deep learning.

What changes is the **model**. In the next lesson, we'll build our first neural network: layers of linear transformations separated by non-linear activation functions. Same gradient descent training loop, but now the model can learn curves, interactions, and patterns that linear regression never could.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>